In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [2]:
patients = pd.read_csv('../data/raw/patients.csv', parse_dates=['registration_date'])
vitals = pd.read_csv('../data/raw/vital_signs.csv', parse_dates=['timestamp'])
history = pd.read_csv('../data/raw/clinical_history.csv')
labs = pd.read_csv('../data/raw/laboratory_results.csv', parse_dates=['timestamp'])
outcomes = pd.read_csv('../data/raw/sepsis_outcomes.csv', parse_dates=['diagnosis_time'])



tables = {'patients': patients, 'vitals': vitals, 'history': history, 'labs': labs, 'outcomes': outcomes}
for name, df in tables.items():
    print(f"{name:10s} Shape={df.shape}")

patients   Shape=(600, 5)
vitals     Shape=(11807, 8)
history    Shape=(1449, 6)
labs       Shape=(2430, 8)
outcomes   Shape=(600, 6)


In [3]:
vitals.columns


Index(['observation_id', 'patient_id', 'timestamp', 'heart_rate',
       'temperature', 'oxygen_saturation', 'respiratory_rate',
       'blood_pressure'],
      dtype='str')

In [4]:
labs.columns

Index(['lab_id', 'patient_id', 'timestamp', 'white_cell_count', 'crp',
       'lactate', 'creatinine', 'platelet_count'],
      dtype='str')

In [5]:
vitals_col = ['heart_rate',
       'temperature', 'oxygen_saturation', 'respiratory_rate',
       'blood_pressure']

labs_col = ['white_cell_count', 'crp',
       'lactate', 'creatinine', 'platelet_count']

In [6]:
outcomes.columns

Index(['outcome_id', 'patient_id', 'sepsis_event', 'diagnosis_time',
       'hospitalisation_required', 'outcome_status'],
      dtype='str')

In [7]:
rng = np.random.default_rng(7)

def get_prediction_time(row, vitals_df):
    if row['sepsis_event']:
        return row['diagnosis_time'] - pd.Timedelta(hours=9)
    pv = vitals_df[vitals_df['patient_id'] == row['patient_id']]
    start, end =pv['timestamp'].min(), pv['timestamp'].max()
    span_hours = max((end - start).total_seconds() / 3600, 1)
    offset = rng.uniform(0.4, 0.9) * span_hours
    return start + pd.Timedelta(hours=offset)


outcomes = outcomes.copy()
outcomes['prediction_time'] = outcomes.apply(get_prediction_time, vitals_df=vitals, axis=1)
outcomes[['patient_id', 'sepsis_event', 'diagnosis_time', 'prediction_time']].head(15)

,patient_id,sepsis_event,diagnosis_time,prediction_time
0,1,False,NaT,2024-10-21 22:31:02.552633471
1,2,False,NaT,2025-06-26 10:29:09.302720280
2,3,False,NaT,2024-03-01 02:25:35.501601845
3,4,False,NaT,2024-08-02 16:10:19.745594826
4,5,False,NaT,2024-07-10 00:00:48.032376176
5,6,False,NaT,2024-10-17 12:01:52.909136091
6,7,False,NaT,2025-12-17 07:05:33.790790426
7,8,False,NaT,2024-08-30 14:15:42.264998141
8,9,True,2024-06-01 22:58:00,2024-06-01 13:58:00.000000000
9,10,False,NaT,2024-03-31 10:06:58.001875856


In [8]:
LOOKBACK_HOURS = 6

def vital_features(pid, cutoff, df):
    window = df[
        (df['patient_id'] == pid) &
        (df['timestamp'] <= cutoff) &
        (df['timestamp'] >= cutoff - pd.Timedelta(hours=LOOKBACK_HOURS))
    ]
    if window.empty:
        window = df[(df['patient_id'] == pid) & (df['timestamp'] <= cutoff)].tail(1)
    feats = {}
    for col in vitals_col:
        vals = window[col]
        feats[f'{col}_mean'] = vals.mean()
        feats[f'{col}_min'] = vals.min()
        feats[f'{col}_max'] = vals.max()
        feats[f'{col}_std'] = vals.std() if len(vals) > 1 else 0.0
        feats[f'{col}_last'] = vals.iloc[-1] if len(vals) else np.nan

        if len(window) > 1:
            hours = (window['timestamp'].iloc[-1] - window['timestamp'].iloc[0]).total_seconds() / 3600
            feats[f'{col}_rate_per_hr'] = (vals.iloc[-1] - vals.iloc[0]) / hours if hours > 0 else 0.0
        else:
            feats[f'{col}_rate_per_hr'] = 0.0
        
    return feats
        



In [9]:
vital_feature_rows = [
    {'patient_id': pid, **vital_features(pid, cutoff, vitals)}
    for pid, cutoff in zip(outcomes['patient_id'], outcomes['prediction_time'])
]

vital_features_df = pd.DataFrame(vital_feature_rows)
vital_features_df.head(10)

,patient_id,heart_rate_mean,heart_rate_min,heart_rate_max,heart_rate_std,heart_rate_last,heart_rate_rate_per_hr,temperature_mean,temperature_min,temperature_max,temperature_std,temperature_last,temperature_rate_per_hr,oxygen_saturation_mean,oxygen_saturation_min,oxygen_saturation_max,oxygen_saturation_std,oxygen_saturation_last,oxygen_saturation_rate_per_hr,respiratory_rate_mean,respiratory_rate_min,respiratory_rate_max,respiratory_rate_std,respiratory_rate_last,respiratory_rate_rate_per_hr,blood_pressure_mean,blood_pressure_min,blood_pressure_max,blood_pressure_std,blood_pressure_last,blood_pressure_rate_per_hr
0,1,84.200000,84.2,84.2,0.000000,84.2,0.000000,36.800,36.80,36.80,0.000000,36.80,0.000000,98.900000,98.9,98.9,0.000000,98.9,0.000000,14.300000,14.3,14.3,0.000000,14.3,0.000000,114.800000,114.8,114.8,0.000000,114.8,0.000000
1,2,87.400000,87.4,87.4,0.000000,87.4,0.000000,37.540,37.54,37.54,0.000000,37.54,0.000000,97.800000,97.8,97.8,0.000000,97.8,0.000000,17.600000,17.6,17.6,0.000000,17.6,0.000000,114.500000,114.5,114.5,0.000000,114.5,0.000000
2,3,75.300000,72.5,78.1,3.959798,78.1,6.000000,36.435,36.37,36.50,0.091924,36.37,-0.139286,95.300000,95.1,95.5,0.282843,95.5,0.428571,15.250000,15.0,15.5,0.353553,15.5,0.535714,130.250000,125.6,134.9,6.576093,125.6,-9.964286
3,4,72.500000,72.5,72.5,0.000000,72.5,0.000000,36.490,36.49,36.49,0.000000,36.49,0.000000,96.900000,96.9,96.9,0.000000,96.9,0.000000,16.400000,16.4,16.4,0.000000,16.4,0.000000,119.000000,119.0,119.0,0.000000,119.0,0.000000
4,5,74.233333,69.9,79.3,3.149391,69.9,-1.616046,36.810,36.71,36.94,0.098234,NaN,NaN,97.316667,96.8,97.9,0.416733,97.2,-0.017192,12.783333,11.7,13.9,0.760044,12.9,-0.034384,138.250000,128.8,148.6,7.109923,128.8,-3.404011
5,6,78.900000,78.9,78.9,0.000000,78.9,0.000000,37.010,37.01,37.01,0.000000,37.01,0.000000,97.300000,97.3,97.3,0.000000,97.3,0.000000,22.000000,22.0,22.0,0.000000,22.0,0.000000,123.200000,123.2,123.2,0.000000,123.2,0.000000
6,7,64.300000,64.3,64.3,0.000000,64.3,0.000000,37.520,37.52,37.52,0.000000,37.52,0.000000,96.200000,96.2,96.2,0.000000,96.2,0.000000,15.000000,15.0,15.0,0.000000,15.0,0.000000,121.100000,121.1,121.1,0.000000,121.1,0.000000
7,8,80.233333,76.9,83.7,3.401960,80.1,0.897196,36.610,36.23,37.04,0.407308,36.56,0.092523,97.366667,96.2,98.6,1.201388,97.3,-0.364486,14.700000,14.2,15.3,0.556776,14.2,-0.308411,124.733333,120.8,128.1,3.682843,128.1,2.046729
8,9,73.000000,69.6,76.4,4.808326,76.4,4.800000,37.215,37.09,37.34,0.176777,37.34,0.176471,97.150000,96.2,98.1,1.343503,96.2,-1.341176,15.700000,14.2,17.2,2.121320,17.2,2.117647,128.150000,120.1,136.2,11.384419,120.1,-11.364706
9,10,84.950000,82.5,87.4,3.464823,82.5,-1.709302,36.195,35.85,36.54,0.487904,35.85,-0.240698,98.150000,98.0,98.3,0.212132,98.3,0.104651,15.850000,14.3,17.4,2.192031,17.4,1.081395,116.450000,114.2,118.7,3.181981,118.7,1.569767


In [10]:
LAB_LOOKBACK_HOURS = 24

def labs_features(pid, cutoff, df):
    window = df[
        (df['patient_id'] == pid) &
        (df['timestamp'] <= cutoff) &
        (df['timestamp'] >= cutoff - pd.Timedelta(hours=LAB_LOOKBACK_HOURS))
    ]
    if window.empty:
        window = df[(df['patient_id'] == pid) & (df['timestamp'] <= cutoff)].tail(1)
    feats = {}
    for col in labs_col:
        vals = window[col]
        feats[f'{col}_mean'] = vals.mean()
        feats[f'{col}_last'] = vals.iloc[-1] if len(vals) else np.nan
      
    return feats
        

In [11]:
labs_features_rows = [ 
    {'patient_id': pid, **labs_features(pid, cutoff, labs)}
    for pid, cutoff in zip(outcomes['patient_id'], outcomes['prediction_time'])
]
labs_features_df = pd.DataFrame(labs_features_rows)
labs_features_df.head(10)


,patient_id,white_cell_count_mean,white_cell_count_last,crp_mean,crp_last,lactate_mean,lactate_last,creatinine_mean,creatinine_last,platelet_count_mean,platelet_count_last
0,1,6.700,NaN,10.250,8.0,0.840,0.85,0.815,0.78,303.0,301.0
1,2,7.715,7.61,15.500,16.9,0.605,0.54,0.870,0.90,187.5,180.0
2,3,8.210,8.21,7.600,7.6,1.160,1.16,1.450,1.45,290.0,290.0
3,4,8.680,8.68,3.900,3.9,NaN,NaN,1.340,1.34,283.0,283.0
4,5,4.792,4.12,6.225,6.7,0.978,0.97,0.550,0.75,247.0,258.0
5,6,7.630,7.63,0.500,0.5,0.620,0.62,1.340,1.34,332.0,332.0
6,7,9.250,9.25,4.100,4.1,0.790,0.79,0.610,0.61,233.0,233.0
7,8,9.840,9.84,4.900,4.9,0.690,0.69,0.860,0.86,243.0,243.0
8,9,9.860,9.86,0.500,0.5,1.240,1.24,0.770,0.77,239.0,239.0
9,10,6.220,6.22,4.200,4.2,1.100,1.10,1.030,1.03,298.0,298.0


In [12]:
patients.columns

Index(['patient_id', 'age', 'gender', 'medical_conditions',
       'registration_date'],
      dtype='str')

In [13]:
static = patients[['patient_id', 'age', 'gender']].copy()

static['comorbidity_count'] = patients['medical_conditions'].apply(
    lambda x: 0 if pd.isna(x) or x == 'None reported'
    else len(x.split(';'))
)

static['gender_Male'] = (static['gender'] == 'Male').astype(int)
static = static.drop(columns=['gender'])
static.head()

,patient_id,age,comorbidity_count,gender_Male
0,1,66,1,1
1,2,42,1,0
2,3,74,1,1
3,4,77,1,0
4,5,25,0,0


In [14]:
features = (
    static
    .merge(vital_features_df, on='patient_id')
    .merge(labs_features_df, on ='patient_id')
    .merge(outcomes[['patient_id', 'sepsis_event']], on='patient_id')
)

features_cols = [
    c for c in features.columns
    if c not in ('patient_id', 'sepsis_event')
]

numeric_cols = features[features_cols].select_dtypes(include='number').columns

features[numeric_cols] = features[numeric_cols].fillna(
    features[numeric_cols].median()
)

features['sepsis_event'] = features['sepsis_event'].astype(int)

print(features.shape)


(600, 45)


In [15]:
features.head(10)

,patient_id,age,comorbidity_count,gender_Male,heart_rate_mean,heart_rate_min,heart_rate_max,heart_rate_std,heart_rate_last,heart_rate_rate_per_hr,temperature_mean,temperature_min,temperature_max,temperature_std,temperature_last,temperature_rate_per_hr,oxygen_saturation_mean,oxygen_saturation_min,oxygen_saturation_max,oxygen_saturation_std,oxygen_saturation_last,oxygen_saturation_rate_per_hr,respiratory_rate_mean,respiratory_rate_min,respiratory_rate_max,respiratory_rate_std,respiratory_rate_last,respiratory_rate_rate_per_hr,blood_pressure_mean,blood_pressure_min,blood_pressure_max,blood_pressure_std,blood_pressure_last,blood_pressure_rate_per_hr,white_cell_count_mean,white_cell_count_last,crp_mean,crp_last,lactate_mean,lactate_last,creatinine_mean,creatinine_last,platelet_count_mean,platelet_count_last,sepsis_event
0,1,66,1,1,84.200000,84.2,84.2,0.000000,84.2,0.000000,36.800,36.80,36.80,0.000000,36.80,0.000000,98.900000,98.9,98.9,0.000000,98.9,0.000000,14.300000,14.3,14.3,0.000000,14.3,0.000000,114.800000,114.8,114.8,0.000000,114.8,0.000000,6.700,7.585,10.250,8.0,0.840,0.85,0.815,0.78,303.0,301.0,0
1,2,42,1,0,87.400000,87.4,87.4,0.000000,87.4,0.000000,37.540,37.54,37.54,0.000000,37.54,0.000000,97.800000,97.8,97.8,0.000000,97.8,0.000000,17.600000,17.6,17.6,0.000000,17.6,0.000000,114.500000,114.5,114.5,0.000000,114.5,0.000000,7.715,7.610,15.500,16.9,0.605,0.54,0.870,0.90,187.5,180.0,0
2,3,74,1,1,75.300000,72.5,78.1,3.959798,78.1,6.000000,36.435,36.37,36.50,0.091924,36.37,-0.139286,95.300000,95.1,95.5,0.282843,95.5,0.428571,15.250000,15.0,15.5,0.353553,15.5,0.535714,130.250000,125.6,134.9,6.576093,125.6,-9.964286,8.210,8.210,7.600,7.6,1.160,1.16,1.450,1.45,290.0,290.0,0
3,4,77,1,0,72.500000,72.5,72.5,0.000000,72.5,0.000000,36.490,36.49,36.49,0.000000,36.49,0.000000,96.900000,96.9,96.9,0.000000,96.9,0.000000,16.400000,16.4,16.4,0.000000,16.4,0.000000,119.000000,119.0,119.0,0.000000,119.0,0.000000,8.680,8.680,3.900,3.9,1.000,1.00,1.340,1.34,283.0,283.0,0
4,5,25,0,0,74.233333,69.9,79.3,3.149391,69.9,-1.616046,36.810,36.71,36.94,0.098234,36.80,0.000000,97.316667,96.8,97.9,0.416733,97.2,-0.017192,12.783333,11.7,13.9,0.760044,12.9,-0.034384,138.250000,128.8,148.6,7.109923,128.8,-3.404011,4.792,4.120,6.225,6.7,0.978,0.97,0.550,0.75,247.0,258.0,0
5,6,37,0,0,78.900000,78.9,78.9,0.000000,78.9,0.000000,37.010,37.01,37.01,0.000000,37.01,0.000000,97.300000,97.3,97.3,0.000000,97.3,0.000000,22.000000,22.0,22.0,0.000000,22.0,0.000000,123.200000,123.2,123.2,0.000000,123.2,0.000000,7.630,7.630,0.500,0.5,0.620,0.62,1.340,1.34,332.0,332.0,0
6,7,63,1,0,64.300000,64.3,64.3,0.000000,64.3,0.000000,37.520,37.52,37.52,0.000000,37.52,0.000000,96.200000,96.2,96.2,0.000000,96.2,0.000000,15.000000,15.0,15.0,0.000000,15.0,0.000000,121.100000,121.1,121.1,0.000000,121.1,0.000000,9.250,9.250,4.100,4.1,0.790,0.79,0.610,0.61,233.0,233.0,0
7,8,55,0,1,80.233333,76.9,83.7,3.401960,80.1,0.897196,36.610,36.23,37.04,0.407308,36.56,0.092523,97.366667,96.2,98.6,1.201388,97.3,-0.364486,14.700000,14.2,15.3,0.556776,14.2,-0.308411,124.733333,120.8,128.1,3.682843,128.1,2.046729,9.840,9.840,4.900,4.9,0.690,0.69,0.860,0.86,243.0,243.0,0
8,9,60,1,1,73.000000,69.6,76.4,4.808326,76.4,4.800000,37.215,37.09,37.34,0.176777,37.34,0.176471,97.150000,96.2,98.1,1.343503,96.2,-1.341176,15.700000,14.2,17.2,2.121320,17.2,2.117647,128.150000,120.1,136.2,11.384419,120.1,-11.364706,9.860,9.860,0.500,0.5,1.240,1.24,0.770,0.77,239.0,239.0,1
9,10,45,1,1,84.950000,82.5,87.4,3.464823,82.5,-1.709302,36.195,35.85,36.54,0.487904,35.85,-0.240698,98.150000,98.0,98.3,0.212132,98.3,0.104651,15.850000,14.3,17.4,2.192031,17.4,1.081395,116.450000,114.2,118.7,3.181981,118.7,1.569767,6.220,6.220,4.200,4.2,1.100,1.10,1.030,1.03,298.0,298.0,0


In [16]:
features.columns

Index(['patient_id', 'age', 'comorbidity_count', 'gender_Male',
       'heart_rate_mean', 'heart_rate_min', 'heart_rate_max', 'heart_rate_std',
       'heart_rate_last', 'heart_rate_rate_per_hr', 'temperature_mean',
       'temperature_min', 'temperature_max', 'temperature_std',
       'temperature_last', 'temperature_rate_per_hr', 'oxygen_saturation_mean',
       'oxygen_saturation_min', 'oxygen_saturation_max',
       'oxygen_saturation_std', 'oxygen_saturation_last',
       'oxygen_saturation_rate_per_hr', 'respiratory_rate_mean',
       'respiratory_rate_min', 'respiratory_rate_max', 'respiratory_rate_std',
       'respiratory_rate_last', 'respiratory_rate_rate_per_hr',
       'blood_pressure_mean', 'blood_pressure_min', 'blood_pressure_max',
       'blood_pressure_std', 'blood_pressure_last',
       'blood_pressure_rate_per_hr', 'white_cell_count_mean',
       'white_cell_count_last', 'crp_mean', 'crp_last', 'lactate_mean',
       'lactate_last', 'creatinine_mean', 'creatinine_las

In [17]:
check_cols = ['heart_rate_last', 'oxygen_saturation_last', 'crp_last', 'lactate_last']
features.groupby('sepsis_event')[check_cols].mean()

,heart_rate_last,oxygen_saturation_last,crp_last,lactate_last
sepsis_event,,,,
0,78.237311,97.525379,6.312311,0.991193
1,80.506944,97.280556,7.798611,1.040556


In [18]:
corr = features[features_cols + ['sepsis_event']].corr()['sepsis_event'].drop('sepsis_event')
corr.sort_values(key=abs, ascending=False).head(10)

oxygen_saturation_std    0.139567
heart_rate_std           0.138265
blood_pressure_std       0.134113
platelet_count_mean     -0.121564
platelet_count_last     -0.117011
respiratory_rate_last    0.116941
crp_last                 0.115262
blood_pressure_last     -0.115065
respiratory_rate_std     0.108521
blood_pressure_min      -0.106690
Name: sepsis_event, dtype: float64

In [28]:
features.to_csv('../data/processed/sepsis_features.csv', index=False)

In [20]:
print(features.shape)
print(features.columns.tolist())

cur = pd.read_csv('../data/processed/sepsis2.csv')
print(cur.shape)
print(cur.columns.tolist())

print(patients['gender'].value_counts(dropna=False))

(600, 45)
['patient_id', 'age', 'comorbidity_count', 'gender_Male', 'heart_rate_mean', 'heart_rate_min', 'heart_rate_max', 'heart_rate_std', 'heart_rate_last', 'heart_rate_rate_per_hr', 'temperature_mean', 'temperature_min', 'temperature_max', 'temperature_std', 'temperature_last', 'temperature_rate_per_hr', 'oxygen_saturation_mean', 'oxygen_saturation_min', 'oxygen_saturation_max', 'oxygen_saturation_std', 'oxygen_saturation_last', 'oxygen_saturation_rate_per_hr', 'respiratory_rate_mean', 'respiratory_rate_min', 'respiratory_rate_max', 'respiratory_rate_std', 'respiratory_rate_last', 'respiratory_rate_rate_per_hr', 'blood_pressure_mean', 'blood_pressure_min', 'blood_pressure_max', 'blood_pressure_std', 'blood_pressure_last', 'blood_pressure_rate_per_hr', 'white_cell_count_mean', 'white_cell_count_last', 'crp_mean', 'crp_last', 'lactate_mean', 'lactate_last', 'creatinine_mean', 'creatinine_last', 'platelet_count_mean', 'platelet_count_last', 'sepsis_event']
(5000, 45)
['patient_id', 'a

In [27]:
print(cur.shape)
print(cur.columns.tolist())

(5000, 45)
['patient_id', 'age', 'comorbidity_count', 'gender_Male', 'heart_rate_mean', 'heart_rate_min', 'heart_rate_max', 'heart_rate_std', 'heart_rate_last', 'heart_rate_rate_per_hr', 'temperature_mean', 'temperature_min', 'temperature_max', 'temperature_std', 'temperature_last', 'temperature_rate_per_hr', 'oxygen_saturation_mean', 'oxygen_saturation_min', 'oxygen_saturation_max', 'oxygen_saturation_std', 'oxygen_saturation_last', 'oxygen_saturation_rate_per_hr', 'respiratory_rate_mean', 'respiratory_rate_min', 'respiratory_rate_max', 'respiratory_rate_std', 'respiratory_rate_last', 'respiratory_rate_rate_per_hr', 'blood_pressure_mean', 'blood_pressure_min', 'blood_pressure_max', 'blood_pressure_std', 'blood_pressure_last', 'blood_pressure_rate_per_hr', 'white_cell_count_mean', 'white_cell_count_last', 'crp_mean', 'crp_last', 'lactate_mean', 'lactate_last', 'creatinine_mean', 'creatinine_last', 'platelet_count_mean', 'platelet_count_last', 'sepsis_event']
